# SPINE-GPE v7 — Phase 1 Extended Evidence v1.0.0

Segundo pacote da Fase 1. O notebook:

1. valida o Foundation Core congelado;
2. inspeciona candidatos e gera o contrato de variáveis;
3. aguarda revisão explícita do contrato;
4. estima perfis survey-weighted, harmoniza valores e cria comparações 2022–2024;
5. gera lock e freeze próprios.

Nenhum artefato upstream é sobrescrito e não há pooling de microdados.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
from pathlib import Path
from google.colab import files
import hashlib, json, os, shutil, subprocess, sys, zipfile
import pandas as pd

ROOT = Path('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7')
PACKAGE_NAME = 'SPINE_GPEv7_PHASE1_EXTENDED_EVIDENCE_PACKAGE_v1.0.0.zip'
PACKAGE_ZIP = Path('/content') / PACKAGE_NAME
INSTALL_DIR = ROOT / 'scripts' / 'phase1_extended_evidence_v100'

if not PACKAGE_ZIP.is_file():
    print(f'Pacote não encontrado em {PACKAGE_ZIP}. Selecione o ZIP.')
    uploaded = files.upload()
    assert PACKAGE_NAME in uploaded, f'Arquivo esperado: {PACKAGE_NAME}; enviados: {list(uploaded)}'

assert PACKAGE_ZIP.is_file(), PACKAGE_ZIP
print('Pacote localizado:', PACKAGE_ZIP)
print('Tamanho:', PACKAGE_ZIP.stat().st_size, 'bytes')

In [ ]:
if INSTALL_DIR.exists():
    shutil.rmtree(INSTALL_DIR)
INSTALL_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(PACKAGE_ZIP) as zf:
    zf.extractall(INSTALL_DIR)

manifest_path = INSTALL_DIR / 'PACKAGE_MANIFEST_SPINE_GPEv7_PHASE1_EXTENDED_EVIDENCE_v1.0.0.json'
assert manifest_path.is_file(), manifest_path
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))

def sha256_file(path, chunk_size=8*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

failures=[]
for rec in manifest['files']:
    path=INSTALL_DIR / rec['path']
    if not path.is_file() or sha256_file(path) != rec['sha256']:
        failures.append(str(path))
assert not failures, failures
print('Pacote instalado e manifest verificado:', INSTALL_DIR)
print('Arquivos verificados:', len(manifest['files']))

In [ ]:
ENGINE = INSTALL_DIR / 'SPINE_GPEv7_PHASE1_EXTENDED_EVIDENCE_v1.0.0.py'
CORE_LOCK = ROOT / '00_admin' / 'SPINE_GPE_PHASE1_EVIDENCE_LOCK.json'
CORE_FREEZE = ROOT / '00_admin' / 'SPINE_GPE_PHASE1_EVIDENCE_FREEZE.json'
assert ENGINE.is_file(), ENGINE
assert CORE_LOCK.is_file(), CORE_LOCK
assert CORE_FREEZE.is_file(), CORE_FREEZE
CORE_LOCK_SHA = sha256_file(CORE_LOCK)
CORE_FREEZE_SHA = sha256_file(CORE_FREEZE)
print('Phase 1 Core Lock SHA-256:', CORE_LOCK_SHA)
print('Phase 1 Core Freeze SHA-256:', CORE_FREEZE_SHA)

In [ ]:
def run_stream(cmd):
    print(' '.join(map(str, cmd)))
    proc=subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    code=proc.wait()
    print('Exit code:', code)
    return code

AUDIT_RUN_ID='phase1_extended_audit_v100'
cmd=[sys.executable, str(ENGINE), '--root', str(ROOT), '--mode', 'audit', '--run-id', AUDIT_RUN_ID,
     '--expected-phase1-lock-sha256', CORE_LOCK_SHA,
     '--expected-phase1-freeze-sha256', CORE_FREEZE_SHA, '--strict']
assert run_stream(cmd) == 0

In [ ]:
INSPECT_RUN_ID='phase1_extended_inspect_v100'
cmd=[sys.executable, str(ENGINE), '--root', str(ROOT), '--mode', 'inspect', '--run-id', INSPECT_RUN_ID,
     '--expected-phase1-lock-sha256', CORE_LOCK_SHA,
     '--expected-phase1-freeze-sha256', CORE_FREEZE_SHA, '--strict']
assert run_stream(cmd) == 0

SCAFFOLD = ROOT / '05_outputs' / 'tables' / 'phase1_extended_evidence' / f'phase1_extended_variable_contract_scaffold_{INSPECT_RUN_ID}.csv'
CANDIDATES = ROOT / '05_outputs' / 'tables' / 'phase1_extended_evidence' / f'phase1_extended_source_candidates_{INSPECT_RUN_ID}.csv'
assert SCAFFOLD.is_file(), SCAFFOLD
print()
print('Contrato scaffold:', SCAFFOLD)
display(pd.read_csv(SCAFFOLD))
print()
print('Top candidatos:')
display(pd.read_csv(CANDIDATES).groupby('component_id', group_keys=False).head(10))


## Revisão obrigatória do contrato

Revise o scaffold no Drive e salve uma cópia, por exemplo:

```text
05_outputs/tables/phase1_extended_evidence/
phase1_extended_variable_contract_REVIEWED_v100.csv
```

Confirme especialmente:

- arquivo de entrada e SHA-256;
- peso, estrato e UPA;
- filtro do domínio e universo elegível;
- renda mensal e horas semanais;
- informalidade e previdência, com valores verdadeiros;
- sexo, raça/cor, escolaridade e idade;
- geografia;
- `contract_status=READY` somente após revisão.

Preencha também o contrato de deflator para todos os anos monetários. O build bloqueia caso 2020, 2022 e 2024 não estejam cobertos.

In [ ]:
# Ajuste estes caminhos depois da revisão.
SOURCE_CONTRACT = ROOT / '05_outputs' / 'tables' / 'phase1_extended_evidence' / 'phase1_extended_variable_contract_REVIEWED_v100.csv'
DEFLATOR_CONTRACT = ROOT / '05_outputs' / 'tables' / 'phase1_extended_evidence' / 'phase1_monetary_harmonization_contract_REVIEWED_v100.csv'
CATEGORY_LABELS = ROOT / '05_outputs' / 'tables' / 'phase1_extended_evidence' / 'phase1_category_labels_REVIEWED_v100.csv'
RUN_BUILD = False

print('SOURCE_CONTRACT:', SOURCE_CONTRACT)
print('DEFLATOR_CONTRACT:', DEFLATOR_CONTRACT)
print('CATEGORY_LABELS:', CATEGORY_LABELS)
print('RUN_BUILD:', RUN_BUILD)

In [ ]:
if RUN_BUILD:
    assert SOURCE_CONTRACT.is_file(), SOURCE_CONTRACT
    assert DEFLATOR_CONTRACT.is_file(), DEFLATOR_CONTRACT
    reviewed=pd.read_csv(SOURCE_CONTRACT, dtype=str).fillna('')
    assert (reviewed['contract_status'].str.upper() == 'READY').all(), reviewed[['component_id','contract_status']]
    BUILD_RUN_ID='phase1_extended_evidence_final_v100'
    cmd=[sys.executable, str(ENGINE), '--root', str(ROOT), '--mode', 'build', '--run-id', BUILD_RUN_ID,
         '--expected-phase1-lock-sha256', CORE_LOCK_SHA,
         '--expected-phase1-freeze-sha256', CORE_FREEZE_SHA,
         '--source-contract', str(SOURCE_CONTRACT),
         '--deflator-contract', str(DEFLATOR_CONTRACT), '--strict']
    if CATEGORY_LABELS.is_file():
        cmd += ['--category-labels', str(CATEGORY_LABELS)]
    assert run_stream(cmd) == 0
else:
    print('Build não executado. Revise os contratos e defina RUN_BUILD=True.')

In [ ]:
# Verificação final, execute depois do build.
EXT_LOCK = ROOT / '00_admin' / 'SPINE_GPE_PHASE1_EXTENDED_EVIDENCE_LOCK.json'
EXT_FREEZE = ROOT / '00_admin' / 'SPINE_GPE_PHASE1_EXTENDED_EVIDENCE_FREEZE.json'
if EXT_LOCK.is_file() and EXT_FREEZE.is_file():
    lock=json.loads(EXT_LOCK.read_text(encoding='utf-8'))
    freeze=json.loads(EXT_FREEZE.read_text(encoding='utf-8'))
    assert lock['status'] == 'PHASE1_EXTENDED_EVIDENCE_CERTIFIED'
    assert lock['critical_failures'] == []
    assert freeze['status'] == 'FROZEN'
    assert freeze['lock_sha256'] == sha256_file(EXT_LOCK)
    assert Path(freeze['extended_evidence_cube']).is_file()
    assert freeze['extended_evidence_cube_sha256'] == sha256_file(Path(freeze['extended_evidence_cube']))
    print('Extended status:', lock['status'])
    print('Freeze status:', freeze['status'])
    print('Lock SHA-256:', sha256_file(EXT_LOCK))
    print('Freeze SHA-256:', sha256_file(EXT_FREEZE))
    print('Next:', lock['next_phase_or_package'])
else:
    print('Lock/freeze da extensão ainda não existem. Execute o build após revisar os contratos.')